# Fixing Links Notebook

Before running this notebook, make sure to run the following command in the terminal to install the required packages:

```bash
bundle install
bundle exec jekyll build
bundle exec htmlproofer _site --typhoeus-config='{"connecttimeout": 10, "timeout": 30, "max_concurrency": 2}' > htmlproofer-output.txt 2>&1
ruby parse_htmlproofer_log.rb 
```

Each command should be run separately and the final two commands create files for all the htmlproofer errors and warnings. This notebook loads the final csv file to help you see what links exists. You will also need to install the `pandas` library if you haven't already. You can do this by running:

```bash
pip install pandas
```

## Load Libraries and Data

In [2]:
import pandas as pd

In [11]:
df = pd.read_csv("htmlproofer-report.csv")
# Lower case the column names
df.columns = df.columns.str.lower()
print(f"Number of errors: {len(df)}")

Number of errors: 378


In [12]:
message_counts = df.message.value_counts().reset_index()
message_counts[(message_counts['count']>1) & (~message_counts.message.str.contains("HTTPS|https")) & (~message_counts.message.str.contains("does not have an alt attribute"))].to_dict(orient="records")

[{'message': "internally linking to /en/lessons/creating-network-diagrams-from-historical-sources#developing-a-coding-scheme; the file exists, but the hash 'developing-a-coding-scheme' does not",
  'count': 3},
 {'message': "internally linking to /en/lessons/building-static-sites-with-jekyll-github-pages#writing-pages-and-posts; the file exists, but the hash 'writing-pages-and-posts' does not",
  'count': 3},
 {'message': "internally linking to #case-study-2:-text-reuse-in-a-large-corpus-of-historical-newspapers; the file exists, but the hash 'case-study-2:-text-reuse-in-a-large-corpus-of-historical-newspapers' does not",
  'count': 2},
 {'message': 'internally linking to georeferencing-qgis, which does not exist',
  'count': 2},
 {'message': 'internally linking to trabajar-con-paginas-web, which does not exist',
  'count': 2},
 {'message': "internally linking to /en/lessons/jupyter-notebooks#installing-jupyter-notebooks; the file exists, but the hash 'installing-jupyter-notebooks' doe

In [13]:
df[(df.message == "internally linking to georeferencing-qgis, which does not exist") ]

,file,line,message
144,_site/en/lessons/geocoding-qgis/index.html,466,"internally linking to georeferencing-qgis, whi..."
216,_site/en/lessons/vector-layers-qgis/index.html,476,"internally linking to georeferencing-qgis, whi..."


In [49]:
df.file.value_counts()

file
_site/es/lecciones/intro-a-google-maps-y-google-earth/index.html                      40
_site/en/lessons/sonification/index.html                                              13
_site/en/lessons/collaborative-blog-with-jekyll-github/index.html                     11
_site/pt/licoes/som-dados-sonificacao-historiadores/index.html                         6
_site/pt/licoes/geocodificando-qgis/index.html                                         5
                                                                                      ..
_site/en/lessons/creating-mobile-augmented-reality-experiences-in-unity/index.html     1
_site/en/lessons/computer-vision-deep-learning-pt2/index.html                          1
_site/en/lessons/computer-vision-deep-learning-pt1/index.html                          1
_site/en/lessons/choropleth-maps-python-folium/index.html                              1
_site/pt/licoes/transcricao-automatica-grafias-nao-latinas/index.html                  1
Name: count, Len

In [52]:
df[df.file == "_site/es/lecciones/intro-a-google-maps-y-google-earth/index.html"].to_dict(orient="records")

[{'file': '_site/es/lecciones/intro-a-google-maps-y-google-earth/index.html',
  'line': 1686,
  'message': 'internally linking to /images/intro-a-google-maps-y-google-earth/geo-es1.png, which does not exist'},
 {'file': '_site/es/lecciones/intro-a-google-maps-y-google-earth/index.html',
  'line': 1687,
  'message': 'internally linking to /images/intro-a-google-maps-y-google-earth/geo-es2.png, which does not exist'},
 {'file': '_site/es/lecciones/intro-a-google-maps-y-google-earth/index.html',
  'line': 1688,
  'message': 'internally linking to /images/intro-a-google-maps-y-google-earth/geo-es3.png, which does not exist'},
 {'file': '_site/es/lecciones/intro-a-google-maps-y-google-earth/index.html',
  'line': 1689,
  'message': 'internally linking to /images/intro-a-google-maps-y-google-earth/geo-es4.png, which does not exist'},
 {'file': '_site/es/lecciones/intro-a-google-maps-y-google-earth/index.html',
  'line': 1690,
  'message': 'internally linking to /images/intro-a-google-maps-y-

In [9]:
file_counts_df = df.file.value_counts().reset_index()
file_counts_df['count_index'] = file_counts_df.index

file_counts_df

,file,count,count_index
0,_site/assets/from-html-to-list-of-words-1/obo-...,53,0
1,_site/assets/normaliser-donnees-textuelles-pyt...,53,1
2,_site/en/lessons/sonification.html,27,2
3,_site/en/lessons/collaborative-blog-with-jekyl...,23,3
4,_site/pt/licoes/som-dados-sonificacao-historia...,22,4
...,...,...,...
509,_site/assets/mapping-with-python-leaflet/exerc...,2,509
510,_site/assets/mapping-with-python-leaflet/exerc...,2,510
511,_site/assets/mapping-with-python-leaflet/map/m...,2,511
512,_site/assets/mapping-with-python-leaflet/map/m...,2,512


In [10]:
merged_df = df.merge(file_counts_df, on='file', how='outer').sort_values(by="count_index", ascending=True)

In [6]:
merged_df[merged_df.file.str.contains("_site/assets/from-html-to-list-of-words-1/", na=False)].message.value_counts()

message
'a' tag is missing a reference                                                                                              12
internal image i/genericThumb.jpg does not exist                                                                             5
internally linking to static/Contact.jsp, which does not exist                                                               2
internally linking to images.jsp?doc=178006280090, which does not exist                                                      2
internally linking to images.jsp?doc=178006280088, which does not exist                                                      2
internally linking to images.jsp?doc=178006280087, which does not exist                                                      2
internally linking to images.jsp?doc=178006280089, which does not exist                                                      2
internally linking to images.jsp?doc=178006280084, which does not exist                                

In [7]:
merged_df[merged_df.message == "internal image i/genericThumb.jpg does not exist"].file.value_counts()

file
_site/assets/from-html-to-list-of-words-1/obo-t17800628-33.html            5
_site/assets/normaliser-donnees-textuelles-python/obo-t17800628-33.html    5
Name: count, dtype: int64

In [8]:
merged_df[merged_df.file.str.contains("_site/en/lessons/building-static-sites-with-jekyll-github-pages", na=False)]

,file,line,message,count,count_index
412,_site/en/lessons/building-static-sites-with-je...,1334,External link https://support.native-instrumen...,10,88
411,_site/en/lessons/building-static-sites-with-je...,66,External link https://maxcdn.bootstrapcdn.com/...,10,88
410,_site/en/lessons/building-static-sites-with-je...,57,External link https://maxcdn.bootstrapcdn.com/...,10,88
406,_site/en/lessons/building-static-sites-with-je...,117,'a' tag is missing a reference,10,88
413,_site/en/lessons/building-static-sites-with-je...,1474,External link https://jekyllthemes.org/ failed...,10,88
414,_site/en/lessons/building-static-sites-with-je...,1545,External link https://jekyll-windows.juthilo.c...,10,88
415,_site/en/lessons/building-static-sites-with-je...,1548,External link https://chronicle.com/blogs/prof...,10,88
407,_site/en/lessons/building-static-sites-with-je...,136,'a' tag is missing a reference,10,88
408,_site/en/lessons/building-static-sites-with-je...,173,'a' tag is missing a reference,10,88
409,_site/en/lessons/building-static-sites-with-je...,199,'a' tag is missing a reference,10,88


In [30]:
import os
import re

EXTENSIONS = (".yml")

def replace_links_preserving_code_blocks(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Match code blocks (triple backticks) and inline code (`...`)
    code_blocks = list(re.finditer(r"(```.*?```|`[^`]*`)", content, re.DOTALL))
    modified = content
    offset = 0

    for match in code_blocks:
        start, end = match.span()
        segment = content[start:end]

        # Temporarily mark this section to skip
        placeholder = f"%%CODEBLOCK{start}%%"
        modified = modified[:start + offset] + placeholder + modified[end + offset:]
        offset += len(placeholder) - (end - start)

    # Replace all http:// with https://
    modified = re.sub(r"http://", "https://", modified)

    # Restore code blocks untouched
    for match in code_blocks:
        start = match.start()
        placeholder = f"%%CODEBLOCK{start}%%"
        modified = modified.replace(placeholder, match.group(0))

    if content != modified:
        print(f"✅ Updated: {file_path}")
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(modified)

def process_all_files(root="."):
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if fname.endswith(EXTENSIONS) and "ph_authors" in fname:
                replace_links_preserving_code_blocks(os.path.join(dirpath, fname))

process_all_files()

✅ Updated: ./_data/ph_authors.yml
